# EM proofreading — Phase A (annotate) demo

Skeleton-driven fly-through over MICrONS minnie65: review a whole cell and drop
tagged annotations marking proofreading errors. **Read-only** — annotate now, edit
manually later (Phase B), then re-enter on the new root id (Phase C).

Design: [docs/proofreading-workflow.md](docs/proofreading-workflow.md) · vocabulary:
[CONTEXT.md](CONTEXT.md). Run in the `em` env (`uv run --extra em jupyter lab`, or pick
`.venv/bin/python3` as the VS Code kernel); needs a CAVE token at
`~/.cloudvolume/secrets/cave-secret.json`.


## 0. Imports


In [1]:
import proofreading.em as em
from proofreading.em.wal import WAL

## 1. Connect to CAVE

`minnie65_public` is the read-only **sandbox** (no edits, no root changes). Use
`minnie65_phase3_v1` for the live, proofreadable datastack.


In [2]:
client = em.EMClient('minnie65_public')
#   live: em.EMClient('minnie65_phase3_v1', version=<materialization_version>)
print('datastack', client.datastack, '| materialization', client.mat_version)

datastack minnie65_public | materialization 1718


## 2. Start a session

Loads the L2 skeleton, captures the **seed supervoxel** (durable identity), builds the
viewer, and opens/append-resumes the write-ahead log. The defaults below feed the
pre-render review (§5) and the live-glide fallback (§6).

In [3]:
root_id = 864691135572530981   # example cell on minnie65_public
sess = em.ProofreadSession(
    client, root_id, wal_dir='./proofread_sessions',
    # --- defaults below feed both the pre-render (§5) and the live-glide fallback (§6) ---
    orient_to_path=False,           # native axis-aligned XYZ sections (fast); True = cross-section ⊥ neurite
    step_nm=1000.0,                 # node spacing along the path = one rendered frame
    animate=True,                   # live glide: smooth tween between nodes
    seconds_per_step=0.5,           # live glide: tween duration per node
    dwell_seconds=0.5,              # live glide: rest at each node so the seg paints
    cross_section_render_scale=2,   # 1.0 = full res (mip0); 2.0 ~ mip1 (faster)
)
print('seed supervoxel:', sess.seed)
print('branch paths:', len(sess.tree.branch_paths), '| summary:', sess.summary())

seed supervoxel: 111692652428576376
branch paths: 189 | summary: {'to_review': 184, 'covered': 3, 'omitted': 2}


## 3. Open the viewer

Open this URL in a browser — the 4-panel layout (3 cross-sections + 3D). The target
cell is highlighted; other segments are off until you reveal them with `n`.

By default the fly-through keeps the **native axis-aligned XYZ sections** — they stream
fast (native chunk layout, like manual scrolling). To instead orient panel 1 as a
**cross-section ⊥ the neurite** (nicer for judging merges, but oblique slices stream
slower), set `orient_to_path=True` (§2) or `sess.set_orient_to_path(True)` live.


In [4]:
sess.viewer

http://localhost:62093/v/0374dbb18c4d28c524b5b3cd395b25e07f53e478/

## 4. Control panel + key map

The panel shows the branch-path checklist with **Review**, **Mark done**, **Resolve
supervoxels**. While flying a path, use these keys in the neuroglancer window:

| key | action |
|-----|--------|
| `m` | merge error |
| `s` | split error |
| `e` | extend |
| `q` | question |
| `n` | toggle the segment under the cursor (reveal/hide a neighbor) |
| `x` | mark current branch path reviewed (and advance) |

A `merge error` ends the path early and **prunes the distal subtree** off the checklist.
Play / pause / step / reverse + the speed slider live on the FlyThrough controls shown after **Review**.


In [5]:
sess.panel()

## 5. Review = smooth glide over a precomputed local layer, pause to act

**This is the fix for the bottleneck.** For each branch we precompute a small EM (+ red
**target** overlay) volume and serve it as a **local** neuroglancer layer — already in
memory, so it renders *sharply while the camera moves*. The fly-through glides over it
**continuously** (no per-node rest), target visible throughout. When you **pause**, we flip
to the live `img`+`seg` layers (which paint because the camera is now idle) at that exact
spot — drop annotations (`m`/`s`/`e`/`q`), toggle neighbors (`n`), mark done (`x`), resume.

- `review_next()` / `review_path(pid)` builds the branch preview and starts a **paused** glide.
- **Play →** smooth glide on the local EM+target preview. **Pause →** live full-res EM + real seg.
- Knobs: `preview_target_nm` (≈96 nm; coarser = faster/smaller), `preview_pad_nm` (context around the neurite).

The preview is a thin per-branch tube at a coarse mip (single-digit MB), built lazily and
rebuilt as you advance. **Optional one-time sanity check** (next cell): confirm a local
layer really stays crisp during motion before relying on it.

In [2]:
# OPTIONAL one-time sanity check (synthetic, no CAVE): a local layer + flythrough.
# Open spike_viewer, then spike_fly.play() — if the checker stays CRISP while moving
# (no blur/blank), the precomputed-preview approach holds. spike_fly.pause() to stop.
spike_viewer, spike_fly = em.localvolume_spike()
spike_viewer

http://localhost:62024/v/43edd2e83d02a995c6035b0577eff185c11515e5/

In [4]:
fly = sess.review_next()      # builds the branch preview + starts a PAUSED glide
# fly.play()  -> smooth glide over the local EM+target preview (renders during motion)
# fly.pause() -> live img+seg paint here; drop m/s/e/q, toggle n, mark x, then fly.play()
# fly.reverse() / fly.step(1)  ; the preview rebuilds when you advance to the next branch
fly

KeyboardInterrupt: 

## 6. Live glide (fallback, no preview)

If you'd rather not precompute, `review_path(pid, preview=False)` is the old live glide: a
smooth tween with a **rest at each node** so the live seg can paint during the rest (it
can't show seg *while moving*). Slower and jerkier — the preview glide above is the default.
Knobs are live-adjustable.

In [ ]:
pid = sess.coverage.to_review(sess.tree)[0]
fly = sess.review_path(pid, preview=False)   # live-glide fallback (rest at each node)
# live knobs (no restart):
# sess.set_dwell(1.0)            # rest at each node so the seg paints (0 = continuous)
# sess.set_orient_to_path(False) # axis-aligned (fast) vs cross-section ⊥ neurite
# sess.set_render_scale(2.0)     # ~mip1 while flying; 1.0 for fine detail
fly

## 7. Checkpoint — resolve supervoxels

Annotations record the click `xyz` immediately (durable); supervoxels are derived in
batch from CloudVolume. Run at checkpoints / before stepping away.


In [ ]:
print('resolved', sess.resolve_supervoxels(), 'supervoxels')

## 8. Inspect what's recorded

The append-only write-ahead log is the source of truth — replay it any time.


In [ ]:
state = WAL.load(sess.wal.path)
print('log:', sess.wal.path)
for a in state.annotations.values():
    print(f'  {a.tag:12s} xyz={[round(c) for c in a.xyz]} supervoxel={a.supervoxel}')
print('coverage summary:', sess.summary())

## 9. After edits — re-entry (Phase C)

Once you've performed the manual splits/merges, the root id changes. Recover the current
root from the durable seed and start a fresh session: the same WAL resumes, prior coverage
re-attaches by L2 id, and edited regions fall back to `to_review`.

```python
new_root = client.current_root(sess.seed)
sess2 = em.ProofreadSession(client, new_root, wal_dir='./proofread_sessions')
sess2.summary()
```


## 10. Shutdown


In [ ]:
sess.close()